# Shared SSL Phase 1: BIOT Pre-Training

## Overview
This notebook implements the **Self-Supervised Learning (SSL)** pre-training for the **BIOT** architecture. 
It is a **shared step**: the encoder is trained once on unlabeled physiological data (Corponi Multisource) to learn general feature representations. The weights generated here are then used for fine-tuning on both **Hosseini** and **WESAD** datasets.

### Instructions:
1. Run this notebook to generate `biot_encoder_pretrained.pt`.
2. Use the generated weights in the comparison notebooks found in the dataset-specific folders.

In [ ]:
# ── CELL 1: Environment Setup ──────────────────────────────────────────
import os, sys, shutil

WORK_DIR = '/kaggle/working/e4selflearning'

if not os.path.exists(WORK_DIR):
    print("Clonazione repository...")
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

print("Installazione dipendenze di training...")
!pip install --no-cache-dir biosppy flirt ruamel.yaml hrv-analysis neurokit2 distinctipy joypy peakutils pytorch-lightning torchmetrics einops biopsykit

print("\n--- INSTALLAZIONE COMPLETATA ---")
print("Riavvia il kernel (se non lo fa da solo) e prosegui dalla Cella 2.")
import os
os._exit(0)


In [ ]:
# ── CELL 2: Patch & Data Ingestion ──────────────────────────────────────────
import os, sys, glob, shutil, pickle

WORK_DIR = '/kaggle/working/e4selflearning'
if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

# --- PATCH 1: Numpy 2.0 (newshape -> shape) ---
def apply_patch(file_path, old_str, new_str):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f: c = f.read()
        if old_str in c:
            c = c.replace(old_str, new_str)
            with open(file_path, 'w') as f: f.write(c)
            print(f"Patch OK: {file_path}")

apply_patch('segment.py', 'newshape=', 'shape=')
apply_patch('src/timebase/data/preprocessing.py', 'newshape=', 'shape=')

# --- PATCH 2: Aggiunge 'dati_preelaborati' a static.py ---
static_p = 'src/timebase/data/static.py'
if os.path.exists(static_p):
    with open(static_p, 'r') as f: c = f.read()
    if '"dati_preelaborati"' not in c:
        c = c.replace('    "wesd": 12,\n}', '    "wesd": 12,\n    "dati_preelaborati": 13,\n}')
        c = c.replace('    "wesd": "lawngreen",\n}', '    "wesd": "lawngreen",\n    "dati_preelaborati": "steelblue",\n}')
        with open(static_p, 'w') as f: f.write(c)
        print("Patch OK: static.py (dati_preelaborati)")

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
os.makedirs(TARGET_DIR, exist_ok=True)

print("Ricerca dati nel pannello Input...")
meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files: 
    raise FileNotFoundError("ERRORE: metadata.pkl non trovato!")
src_dir = os.path.dirname(meta_files[0])

for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(TARGET_DIR, item)
    if item == 'metadata.pkl': shutil.copy2(s, d) 
    else: os.symlink(s, d)

with open(os.path.join(TARGET_DIR, 'metadata.pkl'), 'rb') as f: meta = pickle.load(f)
new_sessions = {}
for root, dirs, files in os.walk(TARGET_DIR, followlinks=True):
    if 'channels.h5' in files:
        clean_name = os.path.basename(root)
        new_rel_id = os.path.relpath(root, TARGET_DIR).replace('\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(clean_name):
                new_sessions[new_rel_id] = meta['sessions_info'][old_id]; break

meta['sessions_info'] = new_sessions
with open(os.path.join(TARGET_DIR, 'metadata.pkl'), 'wb') as f: pickle.dump(meta, f)
print("Ingestion completata!")

print("\nInizio Segmentazione...")
!PYTHONPATH=src python segment.py --data_dir {TARGET_DIR} --output_dir data/preprocessed/sl640_ss640_unlabelled --segment_length 640 --segmentation_mode 1 --step_size 640 --num_workers 2 --overwrite


### 2. BIOT Model and SSL Task

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from einops import rearrange

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class BIOTEncoder(nn.Module):
    def __init__(self, n_channels=6, seg_len=640, chunk_size=64, d_model=256, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.n_channels, self.chunk_size, self.d_model = n_channels, chunk_size, d_model
        self.n_chunks = seg_len // chunk_size
        self.patch_embed = nn.Linear(chunk_size, d_model); self.channel_emb = nn.Embedding(n_channels, d_model)
        self.pos_enc = SinusoidalPE(d_model, max_len=self.n_chunks * n_channels + 1); self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers); self.norm = nn.LayerNorm(d_model)
    def forward(self, x, return_all_tokens=False):
        B, C, T = x.shape
        x = rearrange(x[:, :, :self.n_chunks * self.chunk_size], 'b c (n p) -> b c n p', p=self.chunk_size)
        x = self.patch_embed(x) + self.channel_emb(torch.arange(C, device=x.device)).unsqueeze(0).unsqueeze(2)
        x = torch.cat([self.cls_token.expand(B, -1, -1), rearrange(x, 'b c n d -> b (c n) d')], dim=1)
        x = self.norm(self.transformer(self.pos_enc(x)))
        return x if return_all_tokens else x[:, 0, :]

class BIOTReconstructor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder; self.head = nn.Linear(encoder.d_model, encoder.chunk_size)
    def forward(self, x):
        feats = self.encoder(x, return_all_tokens=True)[:, 1:, :]
        feats = rearrange(feats, 'b (c n) d -> b c n d', c=self.encoder.n_channels, n=self.encoder.n_chunks)
        return rearrange(self.head(feats), 'b c n p -> b c (n p)')

### 3. Shared SSL Training Loop

In [ ]:
import torch.nn.functional as F
from timebase.data.reader import get_datasets
class Args: dataset='data/preprocessed/sl640_ss640_unlabelled'; e4selflearning=True; batch_size=32; num_workers=2; pretext_task='masked_prediction'; masking_ratio=0.15; lm=3; scaling_mode=2; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); seed=42; filter_collections=None; include_hr=False; reuse_stats=False; exclude_anomalies=False; plot_mode=0; task_mode=0; use_wandb=False; unlabelled_data_resampling_percentage=1.0; verbose=0; split_mode=0; overwrite_masks=False

args = Args(); loader, _, _ = get_datasets(args, summary=None)
encoder = BIOTEncoder(seg_len=640).to(args.device); model = BIOTReconstructor(encoder).to(args.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4); criterion = nn.MSELoss()

def dict_to_tensor(d, dev): 
    tensors = [F.interpolate(d[k].float().to(dev).unsqueeze(1), size=640, mode='linear').squeeze(1) for k in sorted(d.keys())]
    return torch.stack(tensors, dim=1)

print("Starting SSL Pre-training (20 epochs)...")
for epoch in range(1, 21):
    model.train(); total_loss = 0.0
    for batch in loader:
        orig, corr, mask = dict_to_tensor(batch['original'], args.device), dict_to_tensor(batch['corrupted'], args.device), dict_to_tensor(batch['mask'], args.device).bool()
        optimizer.zero_grad()
        loss = torch.sqrt(criterion(model(corr)[mask], orig[mask]))
        loss.backward(); optimizer.step(); total_loss += loss.item()
    print(f"Epoch {epoch:2d}/20 | Loss: {total_loss/len(loader):.4f}")

torch.save(encoder.state_dict(), '/kaggle/working/biot_encoder_pretrained.pt')
print("Pre-trained weights saved: biot_encoder_pretrained.pt")